In [3]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles in this directory:")
print(os.listdir())

Current working directory:
C:\Users\mailt\OneDrive\Documents\ticket-priority-project

Files in this directory:
['.git', '.ipynb_checkpoints', '01_data_prep_and_baseline.ipynb', '02_rules_and_models.ipynb', 'aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'archive', 'log file.txt', 'test.csv', 'tickets_en_clean.csv', 'train.csv', 'val.csv']


In [4]:
from pathlib import Path
import os

PROJECT_DIR = Path(
    r"C:\Users\mailt\OneDrive\Documents\ticket-priority-project"
)

os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())
print("train.csv exists:", (PROJECT_DIR / "train.csv").exists())
print("val.csv exists:", (PROJECT_DIR / "val.csv").exists())

Working directory: C:\Users\mailt\OneDrive\Documents\ticket-priority-project
train.csv exists: True
val.csv exists: True


In [10]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv").fillna("")
val = pd.read_csv("val.csv").fillna("")

print("Training shape:", train.shape)
print("Validation shape:", val.shape)

print(train.head(3))

Training shape: (11436, 2)
Validation shape: (2451, 2)
                                          text_clean priority
0  concern about saas platform performance facing...     high
1  problems with marketing agency s system perfor...     high
2  customer support a potential data breach has b...     high


In [11]:
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score)

train = pd.read_csv('train.csv').fillna('')
val   = pd.read_csv('val.csv').fillna('')
CLASSES = ['low', 'medium', 'high']
ymap = {'low': 0, 'medium': 1, 'high': 2}
ytr, yv = train.priority.map(ymap), val.priority.map(ymap)

results = []
def evaluate(name, y_true, y_pred):
    global results
    results = [r for r in results if r['model'] != name]   # no duplicates on re-run
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro'),
        'macro_recall': recall_score(y_true, y_pred, average='macro'),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted')})

print(train.shape, val.shape)
print(train.priority.value_counts(normalize=True).round(3).to_dict())

(11436, 2) (2451, 2)
{'medium': 0.405, 'high': 0.388, 'low': 0.207}


In [12]:
cv = CountVectorizer(stop_words='english', min_df=5, binary=True)
A = cv.fit_transform(train.text_clean)
vocab = np.array(cv.get_feature_names_out())

top_words = {}
for c in CLASSES:
    mask = (train.priority == c).values
    score, _ = chi2(A, mask.astype(int))
    score = np.nan_to_num(score)
    more_in_c = (np.asarray(A[mask].mean(0)).ravel() >
                 np.asarray(A[~mask].mean(0)).ravel())
    top_words[c] = set(vocab[np.argsort(-(score * more_in_c))[:25]])
    print(c, sorted(top_words[c])[:15])

def rule_predict(text):
    words = set(text.split())
    hits = {c: len(words & top_words[c]) for c in CLASSES}
    best = max(hits.values())
    if best == 0:
        return 'medium'
    for c in ['high', 'low', 'medium']:
        if hits[c] == best:
            return c

pred_rule = val.text_clean.apply(rule_predict).map(ymap)
evaluate('Keyword rules', yv, pred_rule)
pd.DataFrame(results).round(3)            

low ['additional', 'assigned', 'audience', 'barcode', 'bulk', 'committing', 'developments', 'durations', 'hr', 'ideas', 'information', 'inquire', 'intermittent', 'invaluable', 'low']
medium ['28', 'achieve', 'collaboration', 'compatible', 'confirming', 'credentials', 'decisions', 'demonstrate', 'details', 'enhancement', 'existing', 'hubspot', 'improve', 'ineffective', 'instability']
high ['affected', 'affecting', 'crashes', 'critical', 'disruption', 'disruptions', 'downtime', 'failure', 'frequent', 'immediate', 'impacting', 'leading', 'major', 'multiple', 'outage']


,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,Keyword rules,0.425,0.409,0.413,0.409,0.428
